# `n_color_immerse()` — why nematic directors need an immersion-based colormap

`nematics3d.field.n_color_immerse()` maps a three-dimensional nematic director

$$
\mathbf n=(n_x,n_y,n_z),\qquad |\mathbf n|=1,
$$

to an sRGB color.

At first sight this may look like an ordinary visualization problem: a director has three components and a color also has three components, so why not simply map one to the other? The difficulty is that a nematic director is **not an ordinary vector**. The two vectors $\mathbf n$ and $-\mathbf n$ represent exactly the same physical orientation. This identification changes the topology of the orientation space and makes a globally perfect three-dimensional color encoding impossible.

This notebook explains the design logic behind `n_color_immerse()` in detail: what an ideal director colormap would mean, why no ideal three-dimensional color map can satisfy every desirable property simultaneously, why Nematics3D uses an immersion related to Boy's surface, and how perceptual optimization in OKLab leads to the selected vividness-aware map.

## 1. What would an ideal nematic colormap do?

Before choosing a formula, it is useful to state explicitly what we would like a director-to-color map

$$
\mathbf c:[\mathbf n]\mapsto(R,G,B)
$$

to achieve. Here $[\mathbf n]$ denotes the nematic orientation represented by both $\mathbf n$ and $-\mathbf n$.

An ideal map would satisfy the following properties.

- **Nematic consistency.**  
  Opposite vectors represent the same physical state, so they must receive exactly the same color:

  $$
  \boxed{\mathbf c(\mathbf n)=\mathbf c(-\mathbf n).}
  $$

  Otherwise the displayed color would depend on an arbitrary sign convention rather than on the nematic orientation itself.

- **Uniqueness.**  
  After the identification $\mathbf n\sim-\mathbf n$ has been taken into account, two physically different orientations should receive different colors:

  $$
  [\mathbf n_1]\neq[\mathbf n_2]
  \quad\Longrightarrow\quad
  \mathbf c([\mathbf n_1])\neq\mathbf c([\mathbf n_2]).
  $$

  In practical terms, uniqueness means that a displayed color could in principle be decoded back to one and only one nematic orientation.

- **Continuity.**  
  Nearby orientations should have nearby colors. If $[\mathbf n_2]\to[\mathbf n_1]$, then we want

  $$
  \mathbf c([\mathbf n_2])\to\mathbf c([\mathbf n_1]).
  $$

  Without continuity, a perfectly smooth director field could contain artificial color jumps that look like physical discontinuities.

- **Local distinguishability.**  
  Continuity alone is not enough. A nematic orientation has two independent local degrees of freedom, so both local directions of change should remain visible in color space. A small two-dimensional patch of orientation space should not collapse into a one-dimensional curve or a single color. Differentially, we want

  $$
  \operatorname{rank}(D_T\mathbf c)=2,
  $$

  where $D_T\mathbf c$ is the differential restricted to the tangent plane of orientation space.

- **Perceptual usefulness.**  
  Equal Euclidean distances in encoded RGB do not correspond to equal perceived color differences. A scientifically useful map should therefore be judged in a perceptual color space rather than only by algebraic distance in RGB coordinates.

- **Semantic interpretability.**  
  It is useful if the Cartesian axes retain familiar colors:

  $$
  \mathbf e_x\to\mathrm{red},\qquad
  \mathbf e_y\to\mathrm{green},\qquad
  \mathbf e_z\to\mathrm{blue}.
  $$

  These associations make orientation plots easier to interpret without repeatedly consulting a legend.

- **Vividness.**  
  The map should not place a large fraction of orientation space near gray or muddy colors. A map can be mathematically smooth and locally non-degenerate while still being visually weak if much of its image lies near the neutral color axis.

- **Displayability.**  
  Every final color must lie in the sRGB gamut:

  $$
  0\le R,G,B\le1.
  $$

  Gamut membership should be enforced during optimization rather than repaired afterward by clipping, because clipping can change the local geometry of the map.

## 2. The orientation space is $\mathbb{RP}^2$, not $\mathbb S^2$

A unit vector in three dimensions lies on the sphere

$$
\mathbb S^2=\{\mathbf n\in\mathbb R^3:|\mathbf n|=1\}.
$$

For a polar vector, every point on this sphere represents a different state. A nematic director is different because

$$
\mathbf n\sim-\mathbf n.
$$

Every pair of antipodal points on $\mathbb S^2$ therefore represents one physical orientation. The physical orientation space is the quotient

$$
\boxed{
\mathbb S^2/\{\mathbf n\sim-\mathbf n\}
\cong\mathbb{RP}^2.
}
$$

This is the correct domain of any nematic director colormap.

A tempting workaround is to choose one representative from each antipodal pair, for example by forcing $n_z\ge0$, and then color only that hemisphere. This does **not** solve the problem. It introduces a seam: orientations that are arbitrarily close in $\mathbb{RP}^2$ may fall on opposite sides of the chosen sign convention and suddenly receive unrelated colors. A sign convention therefore replaces nematic consistency with an artificial discontinuity.

## 3. Why a globally perfect three-dimensional color map is impossible

A displayed color has three coordinates, so it is natural to hope that the two-dimensional space $\mathbb{RP}^2$ could simply be placed inside three-dimensional color space without overlap.

Mathematically, the combination of nematic consistency, global uniqueness, continuity, and local non-degeneracy asks for an **embedding**

$$
\mathbf c:\mathbb{RP}^2\hookrightarrow\mathbb R^3.
$$

An embedding is a continuous one-to-one map that preserves the local topology of the original space. If such a map existed, the entire nematic orientation space could appear as a non-self-intersecting surface in three-dimensional color coordinates.

But

$$
\boxed{\mathbb{RP}^2\not\hookrightarrow\mathbb R^3.}
$$

This is a topological obstruction, not a failure of a particular RGB formula or numerical optimizer. Restricting the target to the sRGB cube cannot help, because the cube is only a subset of $\mathbb R^3$.

Therefore at least one desirable property has to be relaxed. Nematics3D keeps nematic consistency, continuity, local distinguishability, perceptual usefulness, semantic axis colors, vividness, and valid sRGB output, while deliberately giving up **global uniqueness**.

## 4. From embedding to immersion

Once global uniqueness is relaxed, the appropriate mathematical object is an **immersion**

$$
\mathbf c:\mathbb{RP}^2\looparrowright\mathbb R^3.
$$

An immersion may intersect itself globally, so two sufficiently separated orientations may receive the same color. What it does preserve is local dimensionality:

$$
\boxed{\operatorname{rank}(D_T\mathbf c)=2.}
$$

The distinction is therefore:

- an **embedding** forbids both local collapse and global self-intersection;
- an **immersion** forbids local collapse but allows global self-intersection.

For visualization this is a useful compromise. A globally unique inverse color-to-orientation map is impossible, but small changes of orientation can still produce locally distinguishable changes of color.

Boy's surface is a classical immersion of $\mathbb{RP}^2$ into $\mathbb R^3$. This is the origin of both the construction and the name `n_color_immerse`.

## 5. Start from a Boy-type immersion

Nematics3D starts from a fixed polynomial Boy-type map

$$
\mathbf p_B(\mathbf n)=\bigl(p_1(\mathbf n),p_2(\mathbf n),p_3(\mathbf n)\bigr),
$$

with

$$
\begin{aligned}
p_1&=\frac12\left[(2x^2-y^2-z^2)+2yz(y^2-z^2)+zx(x^2-z^2)+xy(y^2-x^2)\right],\\[4pt]
p_2&=\frac78\left[(y^2-z^2)+zx(z^2-x^2)+xy(y^2-x^2)\right],\\[4pt]
p_3&=\frac18(x+y+z)\left[(x+y+z)^3+4(y-x)(z-y)(x-z)\right].
\end{aligned}
$$

The map is even under simultaneous sign reversal:

$$
\mathbf p_B(\mathbf n)=\mathbf p_B(-\mathbf n),
$$

so the nematic identification is built into the construction.

The displayed color is then obtained through an affine transformation

$$
\boxed{\mathbf c_{\rm sRGB}(\mathbf n)=A\mathbf p_B(\mathbf n)+\mathbf b.}
$$

This separation is important. The Boy-type map supplies the required topology; the affine matrix $A$ and offset $\mathbf b$ determine how that immersed surface is positioned, stretched, rotated, and sheared inside color space. As long as $A$ is nonsingular, the local immersion property is preserved.

## 6. Why optimize in OKLab rather than RGB?

The final output has to be sRGB because plotting libraries and displays use RGB values. But encoded RGB coordinates are not perceptually uniform: a fixed Euclidean displacement in one part of the RGB cube may look much larger or smaller than the same displacement elsewhere.

For that reason the optimization evaluates perceptual quantities in **OKLab**. Let

$$
f(\mathbf n)=\operatorname{OKLab}(\mathbf c_{\rm sRGB}(\mathbf n))=(L,a,b).
$$

Two quantities are especially useful:

- distances in $(L,a,b)$, used to measure how distinguishable nearby colors are and how close the Cartesian axes are to their semantic target colors;
- the OKLab chroma

  $$
  C=\sqrt{a^2+b^2},
  $$

  which measures distance from the neutral gray axis and therefore directly diagnoses dull or muddy regions of the colormap.

## 7. Local metric fidelity

Local distinguishability only asks that the differential have rank two. For visualization we would like more: different local directions in orientation space should be represented with reasonably comparable perceptual sensitivity.

Restrict the derivative of $f$ to the tangent plane of $\mathbb S^2$ and define the induced metric

$$
G=(D_Tf)^T(D_Tf).
$$

The distortion objective used in the optimization is

$$
\boxed{
J_{\rm loc}=
\frac{\langle\operatorname{tr}(G^2)\rangle}{\langle\operatorname{tr}G\rangle^2}-\frac12.
}
$$

Smaller $J_{\rm loc}$ is better. The normalization is important because otherwise one could apparently improve an unnormalized metric simply by scaling the entire color surface. $J_{\rm loc}$ instead measures the shape of the local perceptual metric rather than its overall size.

## 8. Preserve interpretable axis colors

A purely geometric optimum need not be easy to read. We therefore preserve the semantic convention

$$
\mathbf e_x\to\mathrm{red},\qquad
\mathbf e_y\to\mathrm{green},\qquad
\mathbf e_z\to\mathrm{blue}.
$$

The targets are the exact sRGB primaries, but deviations are measured in OKLab rather than RGB. This lets the optimization trade a small perceptual shift of an axis color against much larger improvements over the rest of orientation space.

## 9. The first optimization and the $J_{\rm loc}=0.43$ solution

The first completed OKLab optimization considered two objectives: local metric fidelity and axis fidelity. For a chosen local-distortion threshold $t$, it solved approximately

$$
\min J_{\rm axis}
\qquad\text{subject to}\qquad
J_{\rm loc}\le t,
$$

together with the sRGB gamut constraint. Scanning $t$ produced a Pareto frontier, and the knee of that two-objective frontier occurred near

$$
J_{\rm loc}=0.43.
$$

That was a defensible answer to the question *how should local metric fidelity be traded against axis fidelity?* However, it was not yet a satisfactory answer to the broader visualization problem.

The reason is simple: **vividness was not part of the objective**. The optimizer cannot prefer colorful solutions unless colorfulness is explicitly represented. Visual inspection of the $0.43$ map showed that too much of orientation space lay near the low-chroma, gray region of OKLab.

## 10. Revised optimization: maximize vividness under controlled distortion

The revised problem asks a different question:

> How much vividness can be gained while retaining controlled local metric distortion, recognizable Cartesian axis colors, and guaranteed sRGB displayability?

The objective becomes mean OKLab chroma:

$$
\boxed{
\begin{aligned}
\max_{A,\mathbf b}\quad&\langle C(\mathbf n)\rangle\\
\text{s.t.}\quad&J_{\rm loc}\le t,\\
&d_{\rm OKLab}(\mathbf c(\mathbf e_x),\mathrm{red})\le\delta_{\rm axis},\\
&d_{\rm OKLab}(\mathbf c(\mathbf e_y),\mathrm{green})\le\delta_{\rm axis},\\
&d_{\rm OKLab}(\mathbf c(\mathbf e_z),\mathrm{blue})\le\delta_{\rm axis},\\
&\mathbf c_{\rm sRGB}(\mathbf n)\in[0,1]^3\quad\forall[\mathbf n]\in\mathbb{RP}^2.
\end{aligned}
}
$$

The axis tolerance is calibrated from the previous production map rather than chosen without scale. The old map's $x$-axis color had OKLab distance about

$$
\boxed{\delta_{\rm axis}=0.051845}
$$

from exact sRGB red. This already looked acceptably red in practice, so the same perceptual error is used as the maximum allowed deviation for each semantic axis.

## 11. Why the selected solution uses $J_{\rm loc}\le0.55$

Once chroma is promoted to the objective, the meaning of the distortion threshold changes. Increasing $t$ allows the immersed color surface to distort more strongly in perceptual space, which gives the optimizer more freedom to move area away from the neutral axis and toward vivid colors.

The important comparison is therefore not simply '0.43 versus 0.55 as two values of the same optimization.' They belong to different design stages:

- **0.43** is the knee of the earlier two-objective problem involving local metric fidelity and axis fidelity;
- **0.55** is the selected operating point of the revised vividness-aware problem, where mean chroma is maximized under explicit local-distortion, axis-color, and gamut constraints.

The $0.55$ solution provides a substantial increase in visual vividness while keeping local distortion controlled and the $x/y/z$ colors readily recognizable. Larger thresholds can gain more chroma, but increasingly sacrifice local metric fidelity. The selected $0.55$ map is therefore a deliberate balance rather than a mathematically unique optimum independent of visualization goals.

## 12. Why gamut is a hard constraint

A tempting workflow is to optimize freely in color space and then clip the final RGB values into $[0,1]$. That is undesirable here. If a region of the immersed surface lies outside the cube, clipping can flatten an entire patch onto one face or edge of the gamut. This changes the local differential and may create exactly the kind of collapse the immersion was designed to avoid.

Instead, gamut membership is treated as part of the optimization itself. The numerical protocol samples orientation space densely, optimizes under those sampled constraints, verifies the resulting candidate on a much denser set, adds any violating directions to the active set, and repeats until dense verification finds no excursion outside the sRGB cube.

This distinction matters conceptually: clipping is only a final floating-point safeguard in the implementation, not the mechanism that makes an out-of-gamut design displayable.

## 13. How to interpret the resulting colors

The most important practical consequences are:

- `n_color_immerse(n)` is **nematic**, not polar: reversing the director does not change the color.
- Smooth variations of the director produce smooth color variations.
- Nearby orientations remain locally distinguishable because the underlying construction is an immersion.
- Red, green, and blue remain useful visual anchors for the Cartesian $x$, $y$, and $z$ directions.
- The map is intentionally vivid over most of orientation space rather than concentrating near gray.
- A color is **not globally invertible** to a unique orientation. Because $\mathbb{RP}^2$ cannot be embedded in three dimensions, some globally separated orientations must share a color somewhere on the immersed surface.

That last point is not a bug in `n_color_immerse()`. It is the unavoidable price of representing the topology of a three-dimensional nematic orientation with only three color coordinates while retaining continuity and local information.

## 14. Basic usage

In [ ]:
import numpy as np
from nematics3d.field import n_color_immerse

directors = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])

colors = n_color_immerse(directors)
colors

The defining nematic symmetry can be checked directly:

In [ ]:
n = np.array([0.2, 0.4, 0.8944271909999159])
np.allclose(n_color_immerse([n]), n_color_immerse([-n]))

## 15. Visualizing the full colormap: the color sphere

A useful way to understand the map is to evaluate `n_color_immerse()` over a dense set of directions on the unit sphere and display each direction using its assigned color. Because antipodal directions share a color, this sphere is a redundant visualization of the underlying $\mathbb{RP}^2$ map, but it is often the most intuitive way to inspect the result.

When reading such a plot, look for three things:

- continuity of color as the direction moves smoothly over the sphere;
- the red/green/blue anchors near the Cartesian axes;
- the absence of large dull regions near gray.

The self-intersection implied by immersion is not visible as a geometric crossing on this direction sphere. Instead, it appears as the unavoidable fact that some globally separated orientations can share the same displayed color.

## 16. Summary

The design of `n_color_immerse()` follows one chain of reasoning:

1. A three-dimensional nematic director lives in $\mathbb{RP}^2$, not on an ordinary oriented sphere.
2. A perfect continuous and globally unique three-coordinate color encoding would require an embedding $\mathbb{RP}^2\hookrightarrow\mathbb R^3$.
3. Such an embedding does not exist.
4. We therefore give up global uniqueness but preserve continuity and local distinguishability by using an immersion.
5. A Boy-type polynomial provides a convenient fixed immersion, and an affine transformation places it in the sRGB cube.
6. The placement is judged perceptually in OKLab rather than geometrically in encoded RGB.
7. Local metric fidelity, semantic axis colors, vividness, and sRGB gamut membership are balanced explicitly.
8. The selected vividness-aware design uses the $J_{\rm loc}\le0.55$ solution.

The resulting map is therefore not an arbitrary collection of colors. It is a deliberately optimized immersion of nematic orientation space into displayable perceptual color space.